# 03 · Gold — Star schema (SCD2 dims + facts)

Operates on `engine_dwh` only (VPN OFF). Reads from `silver.*`, writes to `gold.*`.

## 1. Imports and engine

In [1]:
import pandas as pd
import hashlib
from datetime import date, datetime, timezone, timedelta
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

load_dotenv(override=True)

SUPABASE_URL  = os.getenv("SUPABASE_URL")
SUPABASE_PORT = int(os.getenv("SUPABASE_PORT", 5432))
SUPABASE_DB   = os.getenv("SUPABASE_DB")
SUPABASE_USER = os.getenv("SUPABASE_USER")
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

def make_engine(host, port, db, user, password, sslmode=None):
    return create_engine(URL.create("postgresql+psycopg2", username=user, password=password, host=host, port=port, database=db, query={"sslmode": sslmode} if sslmode else None))

if not all([SUPABASE_URL, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD]):
    raise ValueError("SUPABASE env vars missing")
engine = make_engine(SUPABASE_URL, SUPABASE_PORT, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD, sslmode="require")
run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print("✓ engine ready.")


✓ engine ready.


## 2. SCD Type 2 helpers

In [2]:
def row_hash(df, cols):
    def hash_row(row):
        parts = ["" if pd.isna(v) else str(v) for v in row]
        return hashlib.md5("|".join(parts).encode()).hexdigest()
    return df[cols].apply(hash_row, axis=1)

def _insert_rows(df, gold_table, surrogate_key):
    cols = [c for c in df.columns if c != surrogate_key and not c.startswith("_")]
    with engine.begin() as conn:
        df[cols].where(pd.notna(df[cols]), None).to_sql(gold_table, conn, schema="gold", if_exists="append", index=False)

def load_scd2(df_silver, gold_table, surrogate_key, business_key, attribute_cols, effective_date):
    gold_table_lower = gold_table.lower()
    with engine.connect() as conn:
        try:
            df_gold = pd.read_sql(text(f"SELECT * FROM gold.{gold_table_lower} WHERE date_to IS NULL"), conn)
        except Exception:
            df_gold = pd.DataFrame()
    n_inserted = 0; n_updated = 0
    silver_keys = set(df_silver[business_key].dropna().tolist())
    hash_silver = row_hash(df_silver, attribute_cols); hash_silver.index = df_silver[business_key].values
    if not df_gold.empty:
        gold_keys = set(df_gold[business_key].dropna().tolist())
        common_attr = [c for c in attribute_cols if c in df_gold.columns]
        hash_gold = row_hash(df_gold, common_attr); hash_gold.index = df_gold[business_key].values
    else:
        gold_keys = set(); hash_gold = pd.Series(dtype=str)
    new_keys = silver_keys - gold_keys
    if new_keys:
        df_new = df_silver[df_silver[business_key].isin(new_keys)].copy()
        df_new["version"] = 1; df_new["date_from"] = effective_date; df_new["date_to"] = None
        _insert_rows(df_new, gold_table_lower, surrogate_key)
        n_inserted = len(df_new)
    common_keys = silver_keys & gold_keys
    changed_keys = {k for k in common_keys if hash_silver.get(k) != hash_gold.get(k)}
    if changed_keys:
        keys_list = list(changed_keys)
        with engine.begin() as conn:
            conn.execute(text(f"UPDATE gold.{gold_table_lower} SET date_to = :dt WHERE {business_key} = ANY(:keys) AND date_to IS NULL"), {"dt": effective_date, "keys": keys_list})
        df_upd = df_silver[df_silver[business_key].isin(changed_keys)].copy()
        df_upd["version"] = df_upd[business_key].map(df_gold.set_index(business_key)["version"]) + 1
        df_upd["date_from"] = effective_date; df_upd["date_to"] = None
        _insert_rows(df_upd, gold_table_lower, surrogate_key)
        n_updated = len(df_upd)
    return {"inserted": n_inserted, "updated": n_updated}


## 3. DimDate

In [3]:
with engine.connect() as conn:
    row = conn.execute(text("SELECT MIN(invoicedate), MAX(invoicedate) FROM silver.stg_fact_invoices")).fetchone()
if row and row[0]:
    date_start = date(row[0].year, 1, 1); date_end = date(row[1].year, 12, 31)
else:
    date_start = date(2013, 1, 1); date_end = date(2016, 12, 31)
records = []
current = date_start
while current <= date_end:
    records.append({"datekey": int(current.strftime("%Y%m%d")), "date": current,
                    "year": current.year, "month": current.month, "day": current.day})
    current += timedelta(days=1)
df_date = pd.DataFrame(records)
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dimdate CASCADE"))
    df_date.to_sql("dimdate", conn, schema="gold", if_exists="append", index=False)
print(f"✓ dimdate populated: {len(df_date)} days from {date_start} to {date_end}.")


✓ dimdate populated: 1461 days from 2017-01-01 to 2020-12-31.


## 4. SCD2 dimensions

In [4]:
with engine.connect() as conn:
    df_emp  = pd.read_sql(text("SELECT personid, fullname, issalesperson FROM silver.stg_employees"), conn)
    df_cust = pd.read_sql(text("SELECT customerid, customername, category FROM silver.stg_customers"), conn)
    df_loc  = pd.read_sql(text("SELECT locationid, city, state, country, salesterritory FROM silver.stg_locations"), conn)
    df_prod = pd.read_sql(text("SELECT stockitemid, stockitemname, brand FROM silver.stg_products"), conn)
    df_dm   = pd.read_sql(text("SELECT deliverymethodid, deliverymethodname FROM silver.stg_deliverymethods"), conn)
    df_pm   = pd.read_sql(text("SELECT paymentmethodid, paymentmethodname FROM silver.stg_paymentmethods"), conn)

s1 = load_scd2(df_emp,  "dimemployee",       "employeekey",       "personid",         ["fullname", "issalesperson"], run_at)
s2 = load_scd2(df_cust, "dimcustomer",       "customerkey",       "customerid",       ["customername", "category"], run_at)
s3 = load_scd2(df_loc,  "dimlocation",       "locationkey",       "locationid",       ["city", "state", "country", "salesterritory"], run_at)
s4 = load_scd2(df_prod, "dimproduct",        "productkey",        "stockitemid",      ["stockitemname", "brand"], run_at)
s5 = load_scd2(df_dm,   "dimdeliverymethod", "deliverymethodkey", "deliverymethodid", ["deliverymethodname"], run_at)
s6 = load_scd2(df_pm,   "dimpaymentmethod",  "paymentmethodkey",  "paymentmethodid",  ["paymentmethodname"], run_at)

print(f"DimEmployee:        {s1}")
print(f"DimCustomer:        {s2}")
print(f"DimLocation:        {s3}")
print(f"DimProduct:         {s4}")
print(f"DimDeliveryMethod:  {s5}")
print(f"DimPaymentMethod:   {s6}")


DimEmployee:        {'inserted': 0, 'updated': 0}
DimCustomer:        {'inserted': 0, 'updated': 0}
DimLocation:        {'inserted': 0, 'updated': 0}
DimProduct:         {'inserted': 0, 'updated': 0}
DimDeliveryMethod:  {'inserted': 0, 'updated': 0}
DimPaymentMethod:   {'inserted': 0, 'updated': 0}


## 5. Facts — FactSales and FactInvoices

Surrogate-key lookups come from current dim rows (`date_to IS NULL`). Unmapped business keys are reported but still load as NULL — the quality script catches them with the unmapped-FK check.

In [5]:
def get_dim_keys(conn, table, business_key, surrogate_key):
    df = pd.read_sql(text(f"SELECT {business_key}, {surrogate_key} FROM gold.{table} WHERE date_to IS NULL"), conn)
    return df.set_index(business_key)[surrogate_key].to_dict()

with engine.connect() as conn:
    df_f_sales = pd.read_sql(text("SELECT * FROM silver.stg_fact_sales"), conn)
    df_f_inv   = pd.read_sql(text("SELECT * FROM silver.stg_fact_invoices"), conn)
    emp_map  = get_dim_keys(conn, "dimemployee",       "personid",          "employeekey")
    cust_map = get_dim_keys(conn, "dimcustomer",       "customerid",        "customerkey")
    loc_map  = get_dim_keys(conn, "dimlocation",       "locationid",        "locationkey")
    prod_map = get_dim_keys(conn, "dimproduct",        "stockitemid",       "productkey")
    dm_map   = get_dim_keys(conn, "dimdeliverymethod", "deliverymethodid",  "deliverymethodkey")

def warn_unmapped(df, biz_col, key_col, label):
    unmapped = df[biz_col].notna() & df[key_col].isna()
    if unmapped.any():
        print(f"⚠ {label}: {int(unmapped.sum())} rows with {biz_col} not in dim — will be NULL")

if not df_f_sales.empty:
    df_f_sales["datekey"] = pd.to_datetime(df_f_sales["invoicedate"]).dt.strftime("%Y%m%d").astype(int)
    df_f_sales["employeekey"] = df_f_sales["salespersonpersonid"].map(emp_map)
    df_f_sales["customerkey"] = df_f_sales["customerid"].map(cust_map)
    df_f_sales["productkey"]  = df_f_sales["stockitemid"].map(prod_map)
    df_f_sales["locationkey"] = df_f_sales["deliverycityid"].map(loc_map)
    warn_unmapped(df_f_sales, "salespersonpersonid", "employeekey", "FactSales.employeekey")
    warn_unmapped(df_f_sales, "customerid",          "customerkey", "FactSales.customerkey")
    warn_unmapped(df_f_sales, "stockitemid",         "productkey",  "FactSales.productkey")
    warn_unmapped(df_f_sales, "deliverycityid",      "locationkey", "FactSales.locationkey")
    factsales = df_f_sales[[
        "invoiceid", "datekey", "employeekey", "customerkey", "productkey", "locationkey",
        "quantity", "unitprice", "taxamount", "extendedprice", "lineprofit"
    ]]
    with engine.begin() as conn:
        conn.execute(text("TRUNCATE TABLE gold.factsales CASCADE"))
        factsales.where(pd.notna(factsales), None).to_sql("factsales", conn, schema="gold", if_exists="append", index=False)
    print(f"✓ FactSales: {len(factsales)} loaded.")

if not df_f_inv.empty:
    df_f_inv["datekey"] = pd.to_datetime(df_f_inv["invoicedate"]).dt.strftime("%Y%m%d").astype(int)
    df_f_inv["employeekey"]         = df_f_inv["salespersonpersonid"].map(emp_map)
    df_f_inv["accountsemployeekey"] = df_f_inv["accountspersonid"].map(emp_map)
    df_f_inv["customerkey"]         = df_f_inv["customerid"].map(cust_map)
    df_f_inv["locationkey"]         = df_f_inv["deliverycityid"].map(loc_map)
    df_f_inv["deliverymethodkey"]   = df_f_inv["deliverymethodid"].map(dm_map) if "deliverymethodid" in df_f_inv.columns else None
    warn_unmapped(df_f_inv, "customerid",       "customerkey",        "FactInvoices.customerkey")
    warn_unmapped(df_f_inv, "deliverymethodid", "deliverymethodkey",  "FactInvoices.deliverymethodkey")
    factinvoices = df_f_inv[[
        "invoiceid", "datekey", "employeekey", "customerkey", "locationkey",
        "accountsemployeekey", "deliverymethodkey",
        "invoiceamount", "paymentdelay_days", "outstandingbalance"
    ]]
    with engine.begin() as conn:
        conn.execute(text("TRUNCATE TABLE gold.factinvoices CASCADE"))
        factinvoices.where(pd.notna(factinvoices), None).to_sql("factinvoices", conn, schema="gold", if_exists="append", index=False)
    print(f"✓ FactInvoices: {len(factinvoices)} loaded.")


✓ FactSales: 228265 loaded.
✓ FactInvoices: 70510 loaded.


## 6. Apply indexes and run quality checks

In [6]:
from pathlib import Path
import subprocess
import sys

sql_path = Path("scripts") / "create_indexes_and_constraints.sql"
if sql_path.exists():
    sql = sql_path.read_text(encoding="utf-8")
    try:
        with engine.begin() as conn:
            conn.execute(text(sql))
        print("✓ Indexes and constraints applied.")
    except Exception as e:
        print(f"✗ Index error: {e}")
else:
    print("Index script not found:", sql_path)

print("\nRunning 04_quality_checks.py ...")
res = subprocess.run([sys.executable, "04_quality_checks.py"], check=False)
print("Quality checks exit code:", res.returncode)


✓ Indexes and constraints applied.

Running 04_quality_checks.py ...
Quality checks exit code: 0
